# **Part I - Setup**

In [6]:
# Installing libs
# %pip install -q rouge_score==0.1.2 evaluate datasets nltk transformers accelerate

import os, gc, math, time, json, random
import numpy as np
import pandas as pd
import torch

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# NLTK sentence tokenizer
import nltk
nltk.download("punkt", quiet=True)
# Some environments name the resource "punkt_tab"—ignore if unavailable
try:
    nltk.download("punkt_tab", quiet=True)
except:
    pass

# HF + evaluate
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    T5ForConditionalGeneration,
    AutoModelForCausalLM
)
import evaluate

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cpu'

# **Part II — Dataset Loading & Exploration**

In [12]:
from datasets import load_dataset

ds = load_dataset("abisee/cnn_dailymail", "3.0.0")

train_sample = pd.DataFrame(ds["train"]).sample(n=100, random_state=42).reset_index(drop=True)
test_sample  = pd.DataFrame(ds["test"]).sample(n=50,  random_state=42).reset_index(drop=True)

print(train_sample.shape, test_sample.shape)
display(train_sample.head(1)[["article", "highlights"]])

(100, 3) (50, 3)


,article,highlights
0,Nasa has warned of an impending asteroid pass ...,2004 BL86 will pass about three times the dist...


# **Part III — Summarization with T5**

In [13]:
def batch_generator(iterable, batch_size):
    """Yield slices of size `batch_size` for efficient generation."""
    for start in range(0, len(iterable), batch_size):
        yield iterable[start:start+batch_size]

def summarize_with_t5(texts, model_name="t5-small", max_input_len=512, max_new_tokens=64, batch_size=8):
    """
    Baby notes:
      - Tokenize with 'summarize: ' prefix so T5 knows the task.
      - Use model.generate to produce summaries.
      - Clear CUDA cache to avoid OOM on small GPUs.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    model = T5ForConditionalGeneration.from_pretrained(model_name).to(DEVICE)
    model.eval()

    outputs = []
    with torch.no_grad():
        for batch in batch_generator(texts, batch_size):
            inputs = [f"summarize: {t}" for t in batch]
            enc = tokenizer(
                inputs,
                max_length=max_input_len,
                truncation=True,
                padding=True,
                return_tensors="pt"
            ).to(DEVICE)

            gen_ids = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                do_sample=False,          # greedy for stability; flip to True for variety
                num_beams=4               # a bit of quality boost
            )
            batch_out = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)
            outputs.extend(batch_out)

            # housekeeping for memory
            del enc, gen_ids
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
            gc.collect()

    # final cleanup
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    gc.collect()
    return outputs

In [15]:
# Generate T5-small summaries on the tiny train sample
t5_small_preds = summarize_with_t5(train_sample["article"].tolist(), model_name="t5-small")
train_t5_small = train_sample.copy()
train_t5_small["pred_t5_small"] = t5_small_preds
display(train_t5_small[["article","highlights","pred_t5_small"]].head(3))

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

,article,highlights,pred_t5_small
0,Nasa has warned of an impending asteroid pass ...,2004 BL86 will pass about three times the dist...,"the asteroid, designated 2004 BL86, will pass ..."
1,"BAGHDAD, Iraq (CNN) -- Iraq's most powerful Su...","Iraqi Islamic Party calls Quran incident ""blat...",a sniper section leader used a Quran for targe...
2,By . David Kent . Andy Carroll has taken an un...,Carroll takes to Instagram to post selfie ahea...,the striker is out for four months after teari...


# **Part IV — “Accuracy”**

In [17]:
import re
def normalize_text(s: str) -> str:
    s = s.lower().strip()
    s = re.sub(r"\s+", " ", s)
    return s

def exact_match_accuracy(preds, refs):
    matches = sum(normalize_text(p)==normalize_text(r) for p,r in zip(preds, refs))
    return matches / max(1, len(refs))

acc_t5_small = exact_match_accuracy(train_t5_small["pred_t5_small"], train_t5_small["highlights"])
print(f"Exact-match accuracy (t5-small): {acc_t5_small:.4f}  ← likely ~0.0 (expected)")

Exact-match accuracy (t5-small): 0.0000  ← likely ~0.0 (expected)


# **Part V — ROUGE Metric**

In [18]:
# Load ROUGE metric
rouge = evaluate.load("rouge")

from nltk.tokenize import sent_tokenize

def _split_sentences(text: str) -> str:
    """Put newlines between sentences (ROUGE expects sentence-delimited text)."""
    sents = sent_tokenize(text.strip())
    return "\n".join(sents)

def compute_rouge_score(preds, refs, use_stemmer=True):
    """
    Baby notes:
      - ROUGE-1: unigrams overlap
      - ROUGE-2: bigrams overlap
      - ROUGE-L: longest common subsequence
    """
    preds_p = [_split_sentences(p) for p in preds]
    refs_p  = [_split_sentences(r) for r in refs]
    scores = rouge.compute(predictions=preds_p, references=refs_p, use_stemmer=use_stemmer)
    return scores

In [20]:
rouge_t5_small = compute_rouge_score(train_t5_small["pred_t5_small"], train_t5_small["highlights"])
rouge_t5_small

{'rouge1': np.float64(0.36670917017746707),
 'rouge2': np.float64(0.17234325852821927),
 'rougeL': np.float64(0.2661210125017955),
 'rougeLsum': np.float64(0.346764877456316)}

# **Part VI — ROUGE Intuition Tests**

In [22]:
# 1) Exact match → high scores
refs = ["the sky is blue.", "cats are cute."]
preds_same = ["the sky is blue.", "cats are cute."]
compute_rouge_score(preds_same, refs)

# 2) Empty prediction → near-zero scores
preds_empty = ["", ""]
compute_rouge_score(preds_empty, refs)

# 3) Stemming demo (play with plural/singular)
refs_s = ["The dog plays in the garden."]
preds_s = ["Dogs play in a garden."]
compute_rouge_score(preds_s, refs_s)     # with stemmer True

# 4) N-gram overlap effect
refs_n = ["machine learning is great for data."]
preds_low = ["learning machines are useful."]
preds_high = ["machine learning is great."]
print("Low overlap:", compute_rouge_score(preds_low, refs_n))
print("High overlap:", compute_rouge_score(preds_high, refs_n))

# 5) Symmetry check (ROUGE calculations are symmetric in this implementation context)
a, b = ["short summary"], ["summary short"]
print("A vs B:", compute_rouge_score(a, b))
print("B vs A:", compute_rouge_score(b, a))

Low overlap: {'rouge1': np.float64(0.4), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.2), 'rougeLsum': np.float64(0.2)}
High overlap: {'rouge1': np.float64(0.8), 'rouge2': np.float64(0.7499999999999999), 'rougeL': np.float64(0.8), 'rougeLsum': np.float64(0.8)}
A vs B: {'rouge1': np.float64(1.0), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.5), 'rougeLsum': np.float64(0.5)}
B vs A: {'rouge1': np.float64(1.0), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.5), 'rougeLsum': np.float64(0.5)}


# **Part VII — Compare Small & Large Models (T5-small, T5-base, GPT-2)**

In [23]:
def summarize_with_gpt2(
    texts,
    model_name="gpt2",
    max_input_len=256,
    max_new_tokens=60,
    batch_size=4,
    tldr_prompt="TL;DR:"
):
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    # Ensure a pad token exists
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(model_name).to(DEVICE)
    model.eval()

    preds = []
    with torch.no_grad():
        for batch in batch_generator(texts, batch_size):
            # Build simple prompt: article + "\nTL;DR:"
            prompts = [ (t[:2000] if t else "") + f"\n{tldr_prompt} " for t in batch ]
            enc = tokenizer(
                prompts,
                max_length=max_input_len,
                truncation=True,
                padding=True,
                return_tensors="pt"
            ).to(DEVICE)

            gen_ids = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                do_sample=True,         # a bit of creativity
                temperature=0.7,
                top_p=0.9,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id
            )
            texts_out = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)
            # Extract only the part after TL;DR:
            clean = []
            for full in texts_out:
                part = full.split(f"{tldr_prompt}", 1)
                clean.append(part[-1].strip() if len(part)>1 else full.strip())
            preds.extend(clean)

            del enc, gen_ids
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
            gc.collect()
    return preds

In [26]:
# Generate across models
models_to_try = {
    "t5-small": lambda texts: summarize_with_t5(texts, "t5-small"),
    "t5-base":  lambda texts: summarize_with_t5(texts, "t5-base"),
    "gpt2":     lambda texts: summarize_with_gpt2(texts, "gpt2")
}

results = {}
for name, fn in models_to_try.items():
    print(f"\n=== Generating with {name} ===")
    preds = fn(train_sample["article"].tolist())
    df = train_sample.copy()
    df[f"pred_{name}"] = preds
    results[name] = df
    display(df[["highlights", f"pred_{name}" ]].head(2))


=== Generating with t5-small ===


,highlights,pred_t5-small
0,2004 BL86 will pass about three times the dist...,"the asteroid, designated 2004 BL86, will pass ..."
1,"Iraqi Islamic Party calls Quran incident ""blat...",a sniper section leader used a Quran for targe...



=== Generating with t5-base ===


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


,highlights,pred_gpt2
0,2004 BL86 will pass about three times the dist...,Nasa has warned of an impending asteroid pass ...
1,"Iraqi Islamic Party calls Quran incident ""blat...","BAGHDAD, Iraq (CNN) -- Iraq's most powerful Su..."


In [28]:
# Compute ROUGE per model
def compute_rouge_per_row(df, pred_col, ref_col="highlights"):
    rows = []
    for p, r in zip(df[pred_col], df[ref_col]):
        s = compute_rouge_score([p],[r])
        flat = {k: float(v) for k,v in s.items()}
        rows.append(flat)
    return pd.DataFrame(rows)

per_row_scores = {}
avg_scores = []
for name, df in results.items():
    col = f"pred_{name}"
    s_df = compute_rouge_per_row(df, col)
    s_df.columns = [f"{c}_{name}" for c in s_df.columns]
    per_row_scores[name] = s_df
    # aggregate
    mean_scores = s_df.mean().to_dict()
    mean_scores = {k.replace(f"_{name}",""): v for k,v in mean_scores.items()}
    avg_scores.append({"model": name, **mean_scores})

avg_scores_df = pd.DataFrame(avg_scores).set_index("model")
display(avg_scores_df.style.format("{:.4f}"))

,rouge1,rouge2,rougeL,rougeLsum
model,,,,
t5-small,0.3673,0.1737,0.2665,0.3480
t5-base,0.4045,0.2064,0.3002,0.3822
gpt2,0.2351,0.1125,0.1579,0.2247


# **Part VIII — Side-by-Side Comparison Helpers**

In [29]:
def compare_models(results_dict, n=5):
    """Show first n rows of references + each model's prediction."""
    out = pd.DataFrame()
    out["reference"] = list(results_dict.values())[0]["highlights"]  # same base df
    for name, df in results_dict.items():
        out[f"pred_{name}"] = df[f"pred_{name}"]
    return out.head(n)

display(compare_models(results, n=5))

,reference,pred_t5-small,pred_t5-base,pred_gpt2
0,2004 BL86 will pass about three times the dist...,"the asteroid, designated 2004 BL86, will pass ...",2004 BL86 will pass about three times the dist...,Nasa has warned of an impending asteroid pass ...
1,"Iraqi Islamic Party calls Quran incident ""blat...",a sniper section leader used a Quran for targe...,u.s. soldier's desecration of the holy book re...,"BAGHDAD, Iraq (CNN) -- Iraq's most powerful Su..."
2,Carroll takes to Instagram to post selfie ahea...,the striker is out for four months after teari...,england striker posts glum-looking selfie in h...,By . David Kent . Andy Carroll has taken an un...
3,Pop stars from all over Europe are setting the...,aspiring pop stars from the u.s. and abroad ar...,singers from all over Europe and farther east ...,Los Angeles (CNN) -- Los Angeles has long been...
4,NEW: Young athletes light the Olympic cauldron...,a billion people around the world would be glu...,"""Isles of Wonder"" features tributes to the Bri...",London (CNN) -- Few shows can claim such an au...
